# Build a guideline Q&A assistant

Exercise notebook for the lesson **[Build a guideline Q&A assistant](https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant)** in the *Retrieval-augmented generation (RAG)* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** A guideline question-answering assistant combines the RAG steps: split the guideline into sections, embed them, retrieve the best match for a question, decline if the match is weak, and answer from the retrieved text with a citation. The version built here answers by quoting the source, which keeps every word traceable and runs without an AI model API. Testing it shows a typical failure: quoting too little can drop the most important step, such as how much carbohydrate to give for a low glucose.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### The assistant

Run the protocol block from the chunking lesson first. Installs: pip install sentence-transformers

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Index: one chunk per section, each split into sentences that can be quoted and cited
chunks = []
for i, part in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:], start=1):
    heading, *body = part.strip().splitlines()
    sentences = re.split(r"(?<=\.)\s+", " ".join(body).strip())
    chunks.append({"id": f"s{i}", "heading": heading.lstrip("# "), "sentences": sentences})
chunk_emb = model.encode([c["heading"] + ". " + " ".join(c["sentences"]) for c in chunks],
                         normalize_embeddings=True)

ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)", "biguanide": "biguanide (metformin)"}

def expand(q):
    for short, long in ABBREVIATIONS.items():
        q = re.sub(rf"\b{short}\b", long, q, flags=re.IGNORECASE)
    return q

def answer(question, min_score=0.35, max_sentences=2):
    q = expand(question)
    q_emb = model.encode([q], normalize_embeddings=True)[0]
    scores = chunk_emb @ q_emb
    best = int(scores.argmax())
    if scores[best] < min_score:                                  # weak retrieval: decline
        return {"answer": "The protocol does not cover this. Please ask your care team.",
                "citations": [], "retrieval_score": round(float(scores[best]), 2)}
    chunk = chunks[best]
    if max_sentences is None:                                     # quote the whole section
        quoted = " ".join(chunk["sentences"])
    else:
        sent_scores = model.encode(chunk["sentences"], normalize_embeddings=True) @ q_emb
        top = sorted(np.argsort(-sent_scores)[:max_sentences])    # keep document order
        quoted = " ".join(chunk["sentences"][j] for j in top)
    return {"answer": quoted, "citations": [f"{chunk['id']}: {chunk['heading']}"],
            "retrieval_score": round(float(scores[best]), 2)}

### Trying it out

Continuing from the previous block: two covered questions and one the protocol does not cover.

In [ ]:
for q in ["When should metformin be stopped?",
          "What should I do about a hypo?",
          "What dose of semaglutide should I start?"]:
    r = answer(q)
    print(f"Q: {q}\nA: {r['answer']}\n   citations {r['citations']}  retrieval score {r['retrieval_score']}\n")

# Q: When should metformin be stopped?
# A: Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.
#    citations ['s2: 2. Kidney function and metformin']  retrieval score 0.53
#
# Q: What should I do about a hypo?
# A: Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.
#    citations ['s4: 4. Low blood glucose']  retrieval score 0.63
#
# Q: What dose of semaglutide should I start?
# A: The protocol does not cover this. Please ask your care team.
#    citations []  retrieval score 0.32

### Fixing the incomplete answer

Continuing from the previous blocks: quote the whole section instead of the top two sentences.

In [ ]:
r = answer("What should I do about a hypo?", max_sentences=None)
print(f"A: {r['answer']}\n   citations {r['citations']}")

# A: A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.
#    citations ['s4: 4. Low blood glucose']

## Exercise

**Goal:** Build, test and improve a guideline Q&A assistant over the synthetic protocol.

**Data:** Synthetic protocol in the notebook; small open embedding model all-MiniLM-L6-v2 (Course content; model under Apache 2.0)

### Step 1

Reproduce the assistant and the three example questions.

In [ ]:
# Your code here


### Step 2

Write ten questions, including two out of scope and two using abbreviations, and record each answer.

In [ ]:
# Your code here


### Step 3

Compare max_sentences=2 with whole-section quoting on all ten questions; which answers lose important information?

In [ ]:
# Your code here


### Step 4

Add a second retrieved section when the top two scores are close, and cite both.

In [ ]:
# Your code here


### Step 5

Optional, with an approved model: add a generation step that rewrites the quoted section as an answer, and check it with the citation checker.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant](https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.